# MODEL

In [168]:
from __future__ import annotations

import copy
import math
from dataclasses import asdict, dataclass

import torch
from torch import Tensor, nn

@dataclass(slots=True)
class Seq2SeqConfig:
    # ===== vocab =====
    src_vocab_size: int   
    tgt_vocab_size: int   

    # CPWord encoder embedding（关键！）
    src_vocab_size_list: list[int]     
    src_embedding_size_list: list[int]  

    # ===== model size =====
    d_model: int = 256   
    nhead: int = 8      

    # ===== layers =====
    num_encoder_layers: int = 4
    num_decoder_layers: int = 4

    # ===== FFN =====
    dim_feedforward: int = 1024

    # ===== dropout =====
    dropout: float = 0.1
    activation: str = "relu"

    # ===== position =====
    position_encoding_type: str = "sinusoidal"
    max_source_length: int = 512
    max_target_length: int = 512

    # ===== tokens =====
    pad_token_id: int = 0
    bos_token_id: int = 1
    eos_token_id: int = 2

    def __post_init__(self):
        if self.d_model % self.nhead != 0:
            raise ValueError("d_model must be divisible by nhead.")
        if self.activation not in {"relu", "gelu"}:
            raise ValueError("activation must be relu or gelu.")
        if self.position_encoding_type not in {"sinusoidal", "learned", "alibi"}:
            raise ValueError("position_encoding_type is unsupported.")
        if len(self.src_vocab_size_list) != len(self.src_embedding_size_list):
            raise ValueError("src_vocab_size_list and src_embedding_size_list must match")

    def to_dict(self):
        return asdict(self)


class CPWordEmbedding(nn.Module):
    def __init__(self, vocab_size_list, embbed_size_list, d_model=512, pad_token_id=0):
        super().__init__()

        self.family_embedding = nn.Embedding(vocab_size_list[0], embbed_size_list[0], padding_idx=pad_token_id)
        self.position_embedding = nn.Embedding(vocab_size_list[1], embbed_size_list[1], padding_idx=pad_token_id)
        self.pitch_embedding = nn.Embedding(vocab_size_list[2], embbed_size_list[2], padding_idx=pad_token_id)
        self.duration_embedding = nn.Embedding(vocab_size_list[3], embbed_size_list[3], padding_idx=pad_token_id)
        self.program_embedding = nn.Embedding(vocab_size_list[4], embbed_size_list[4], padding_idx=pad_token_id)
        self.tempo_embedding = nn.Embedding(vocab_size_list[5], embbed_size_list[5], padding_idx=pad_token_id)
        self.time_signature_embedding = nn.Embedding(vocab_size_list[6], embbed_size_list[6], padding_idx=pad_token_id)

        self.in_linear = nn.Linear(sum(embbed_size_list), d_model)

    def forward(self, x):

        family = self.family_embedding(x[..., 0])
        position = self.position_embedding(x[..., 1])
        pitch = self.pitch_embedding(x[..., 2])
        duration = self.duration_embedding(x[..., 3])
        program = self.program_embedding(x[..., 4])
        tempo = self.tempo_embedding(x[..., 5])
        ts = self.time_signature_embedding(x[..., 6])

        x = torch.cat([family, position, pitch, duration, program, tempo, ts], dim=-1)

        return self.in_linear(x)


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_length: int) -> None:
        super().__init__()
        positions = torch.arange(max_length, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10_000.0) / d_model)
        )
        encoding = torch.zeros(max_length, d_model, dtype=torch.float32)
        encoding[:, 0::2] = torch.sin(positions * div_term)
        encoding[:, 1::2] = torch.cos(positions * div_term)
        self.register_buffer("encoding", encoding.unsqueeze(0), persistent=False)

    def forward(self, tokens: Tensor) -> Tensor:
        return self.encoding[:, : tokens.size(1)]


class LearnedPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_length: int) -> None:
        super().__init__()
        self.embedding = nn.Embedding(max_length, d_model)

    def forward(self, tokens: Tensor) -> Tensor:
        positions = torch.arange(tokens.size(1), device=tokens.device).unsqueeze(0)
        return self.embedding(positions)


class ZeroPositionalEncoding(nn.Module):
    def __init__(self, d_model: int) -> None:
        super().__init__()
        self.d_model = d_model

    def forward(self, tokens: Tensor) -> Tensor:
        return torch.zeros((1, tokens.size(1), self.d_model), device=tokens.device)


def build_positional_encoding(encoding_type: str, d_model: int, max_length: int) -> nn.Module:
    if encoding_type == "sinusoidal":
        return SinusoidalPositionalEncoding(d_model, max_length)
    if encoding_type == "learned":
        return LearnedPositionalEncoding(d_model, max_length)
    if encoding_type == "alibi":
        return ZeroPositionalEncoding(d_model)
    raise ValueError(f"Unsupported positional encoding {encoding_type!r}.")


def build_causal_mask(sequence_length: int, device: torch.device | str) -> Tensor:
    return torch.triu(
        torch.ones(sequence_length, sequence_length, dtype=torch.bool, device=device),
        diagonal=1,
    )


def build_alibi_causal_mask(
    *,
    sequence_length: int,
    num_heads: int,
    batch_size: int,
    device: torch.device | str,
) -> Tensor:
    slopes = torch.tensor(
        [2 ** (-(8.0 * (index + 1) / num_heads)) for index in range(num_heads)],
        dtype=torch.float32,
        device=device,
    ).view(num_heads, 1, 1)
    positions = torch.arange(sequence_length, device=device, dtype=torch.float32)
    distance = (positions[:, None] - positions[None, :]).clamp_min(0.0)
    future = positions[:, None] < positions[None, :]
    bias = (-slopes * distance).masked_fill(future.unsqueeze(0), float("-inf"))
    return bias.repeat_interleave(batch_size, dim=0)


def _activation(name: str):
    return torch.relu if name == "relu" else torch.nn.functional.gelu


def _normalize_padding_mask(mask: Tensor | None, attn_mask: Tensor) -> Tensor | None:
    if mask is None or attn_mask.dtype == torch.bool:
        return mask
    return mask.to(dtype=attn_mask.dtype).masked_fill(mask, float("-inf"))


class TransformerDecoderLayer(nn.Module):
    def __init__(
        self,
        *,
        d_model: int,
        nhead: int,
        dim_feedforward: int,
        dropout: float,
        activation: str,
    ) -> None:
        super().__init__()
        self.self_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            dropout=dropout,
            batch_first=True,
        )
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            dropout=dropout,
            batch_first=True,
        )
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.activation = _activation(activation)

    def forward(
        self,
        tgt: Tensor,
        *,
        tgt_causal_mask: Tensor,
        tgt_padding_mask: Tensor | None = None,
        memory: Tensor | None = None,
        memory_padding_mask: Tensor | None = None,
    ) -> Tensor:
        x = tgt
        self_attn_output = self.self_attn(
            x,
            x,
            x,
            attn_mask=tgt_causal_mask,
            key_padding_mask=_normalize_padding_mask(tgt_padding_mask, tgt_causal_mask),
            need_weights=False,
        )[0]
        x = self.norm1(x + self.dropout1(self_attn_output))

        if memory is not None:
            cross_attn_output = self.cross_attn(
                x,
                memory,
                memory,
                key_padding_mask=_normalize_padding_mask(memory_padding_mask, tgt_causal_mask),
                need_weights=False,
            )[0]
            x = self.norm2(x + self.dropout2(cross_attn_output))

        ff_output = self.linear2(self.dropout(self.activation(self.linear1(x))))
        return self.norm3(x + self.dropout3(ff_output))


class TransformerDecoderStack(nn.Module):
    def __init__(
        self,
        *,
        d_model: int,
        nhead: int,
        num_layers: int,
        dim_feedforward: int,
        dropout: float,
        activation: str,
    ) -> None:
        super().__init__()
        layer = TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation=activation,
        )
        self.layers = nn.ModuleList(copy.deepcopy(layer) for _ in range(num_layers))
        self.norm = nn.LayerNorm(d_model)

    def forward(
        self,
        tgt: Tensor,
        *,
        tgt_causal_mask: Tensor,
        tgt_padding_mask: Tensor | None = None,
        memory: Tensor | None = None,
        memory_padding_mask: Tensor | None = None,
    ) -> Tensor:
        x = tgt
        for layer in self.layers:
            x = layer(
                x,
                tgt_causal_mask=tgt_causal_mask,
                tgt_padding_mask=tgt_padding_mask,
                memory=memory,
                memory_padding_mask=memory_padding_mask,
            )
        return self.norm(x)


class TransformerDecoderLM(nn.Module):
    def __init__(self, config: Seq2SeqConfig) -> None:
        super().__init__()
        self.config = config
        self.tgt_embedding = nn.Embedding(
            num_embeddings=config.tgt_vocab_size,
            embedding_dim=config.d_model,
            padding_idx=config.pad_token_id,
        )
        self.position_encoding = build_positional_encoding(
            config.position_encoding_type,
            config.d_model,
            config.max_target_length,
        )
        self.dropout = nn.Dropout(config.dropout)
        self.decoder = TransformerDecoderStack(
            d_model=config.d_model,
            nhead=config.nhead,
            num_layers=config.num_decoder_layers,
            dim_feedforward=config.dim_feedforward,
            dropout=config.dropout,
            activation=config.activation,
        )
        self.output_projection = nn.Linear(config.d_model, config.tgt_vocab_size)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for parameter in self.parameters():
            if parameter.dim() > 1:
                nn.init.xavier_uniform_(parameter)

    def decode(self, input_tokens: Tensor) -> Tensor:
        embeddings = self.tgt_embedding(input_tokens) * math.sqrt(self.config.d_model)
        return self.dropout(embeddings + self.position_encoding(input_tokens))

    def forward(
        self,
        input_tokens: Tensor,
        *,
        padding_mask: Tensor | None = None,
        memory: Tensor | None = None,
        memory_padding_mask: Tensor | None = None,
    ) -> Tensor:
        if padding_mask is not None:
            padding_mask = padding_mask.to(torch.bool)
        if memory_padding_mask is not None:
            memory_padding_mask = memory_padding_mask.to(torch.bool)

        decoded_inputs = self.decode(input_tokens)

        if self.config.position_encoding_type == "alibi":
            causal_mask = build_alibi_causal_mask(
                sequence_length=input_tokens.size(1),
                num_heads=self.config.nhead,
                batch_size=input_tokens.size(0),
                device=input_tokens.device,
            )
        else:
            causal_mask = build_causal_mask(input_tokens.size(1), device=input_tokens.device)

        hidden_states = self.decoder(
            decoded_inputs,
            tgt_causal_mask=causal_mask,
            tgt_padding_mask=padding_mask,
            memory=memory,
            memory_padding_mask=memory_padding_mask,
        )
        return self.output_projection(hidden_states)


class TransformerEncoderLayer(nn.Module):
    def __init__(
        self,
        *,
        d_model: int,
        nhead: int,
        dim_feedforward: int,
        dropout: float,
        activation: str,
    ) -> None:
        super().__init__()
        self.self_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            dropout=dropout,
            batch_first=True,
        )
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.activation = _activation(activation)

    def forward(
        self,
        src: Tensor,
        *,
        src_padding_mask: Tensor | None = None,
    ) -> Tensor:
        x = src
        self_attn_output = self.self_attn(
            x,
            x,
            x,
            attn_mask=None,
            key_padding_mask=src_padding_mask,
            need_weights=False,
        )[0]
        x = self.norm1(x + self.dropout1(self_attn_output))

        ff_output = self.linear2(self.dropout(self.activation(self.linear1(x))))
        x = self.norm2(x + self.dropout2(ff_output))
        return x


class TransformerEncoderStack(nn.Module):
    def __init__(
        self,
        *,
        d_model: int,
        nhead: int,
        num_layers: int,
        dim_feedforward: int,
        dropout: float,
        activation: str,
    ) -> None:
        super().__init__()
        layer = TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation=activation,
        )
        self.layers = nn.ModuleList(copy.deepcopy(layer) for _ in range(num_layers))
        self.norm = nn.LayerNorm(d_model)

    def forward(
        self,
        src: Tensor,
        *,
        src_padding_mask: Tensor | None = None,
    ) -> Tensor:
        x = src
        for layer in self.layers:
            x = layer(x, src_padding_mask=src_padding_mask)
        return self.norm(x)


class TransformerEncoder(nn.Module):
    def __init__(self, config: Seq2SeqConfig) -> None:
        super().__init__()
        self.config = config

        self.embedding = CPWordEmbedding(
            vocab_size_list=config.src_vocab_size_list,
            embbed_size_list=config.src_embedding_size_list,
            d_model=config.d_model,
            pad_token_id=config.pad_token_id,
        )

        self.position_encoding = build_positional_encoding(
            config.position_encoding_type,
            config.d_model,
            config.max_source_length,
        )

        self.dropout = nn.Dropout(config.dropout)

        self.encoder = TransformerEncoderStack(
            d_model=config.d_model,
            nhead=config.nhead,
            num_layers=config.num_encoder_layers,
            dim_feedforward=config.dim_feedforward,
            dropout=config.dropout,
            activation=config.activation,
        )

        self.reset_parameters()

    def reset_parameters(self) -> None:
        for parameter in self.parameters():
            if parameter.dim() > 1:
                nn.init.xavier_uniform_(parameter)

    def encode(self, input_tokens: Tensor) -> Tensor:
        embeddings = self.embedding(input_tokens)
        pos = self.position_encoding(input_tokens)
        return self.dropout(embeddings + pos)

    def forward(
        self,
        input_tokens: Tensor,
        *,
        padding_mask: Tensor | None = None,
    ) -> Tensor:
        if padding_mask is not None:
            if padding_mask.dim() == 3:
                padding_mask = padding_mask[..., 0]
            padding_mask = padding_mask.to(torch.bool)

        encoded_inputs = self.encode(input_tokens)
        hidden_states = self.encoder(
            encoded_inputs,
            src_padding_mask=padding_mask,
        )
        return hidden_states


class TransformerSeq2Seq(nn.Module):
    def __init__(self, config: Seq2SeqConfig) -> None:
        super().__init__()
        self.config = config
        self.encoder = TransformerEncoder(config)
        self.decoder = TransformerDecoderLM(config)

    def forward(
        self,
        *,
        encoder_input_tokens: Tensor,
        decoder_input_tokens: Tensor,
        encoder_padding_mask: Tensor | None = None,
        decoder_padding_mask: Tensor | None = None,
    ) -> Tensor:
        if encoder_padding_mask is not None:
            if encoder_padding_mask.dim() == 3:
                encoder_padding_mask = encoder_padding_mask[..., 0]
            encoder_padding_mask = encoder_padding_mask.to(torch.bool)

        if decoder_padding_mask is not None:
            decoder_padding_mask = decoder_padding_mask.to(torch.bool)

        memory = self.encoder(
            encoder_input_tokens,
            padding_mask=encoder_padding_mask,
        )

        logits = self.decoder(
            decoder_input_tokens,
            padding_mask=decoder_padding_mask,
            memory=memory,
            memory_padding_mask=encoder_padding_mask,
        )
        return logits


class TransformerForConditionalGeneration(nn.Module):
    def __init__(self, config: Seq2SeqConfig) -> None:
        super().__init__()
        self.config = config
        self.model = TransformerSeq2Seq(config)

    def forward(
        self,
        *,
        encoder_input_tokens: Tensor,
        decoder_input_tokens: Tensor,
        labels: Tensor | None = None,
        encoder_padding_mask: Tensor | None = None,
        decoder_padding_mask: Tensor | None = None,
    ):
        if labels is None:
            raise ValueError("labels must be provided for training")

        if decoder_padding_mask is None:
            decoder_padding_mask = decoder_input_tokens.eq(self.config.pad_token_id)

        logits = self.model(
            encoder_input_tokens=encoder_input_tokens,
            decoder_input_tokens=decoder_input_tokens,
            encoder_padding_mask=encoder_padding_mask,
            decoder_padding_mask=decoder_padding_mask,
        )

        loss = torch.nn.functional.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1),
            ignore_index=-100,
        )

        return loss, logits
    
    
class LoRA(nn.Module):
    def __init__(self, in_features, out_features, rank):
        super().__init__()
        self.A = nn.Linear(in_features, rank, bias=False)
        self.B = nn.Linear(rank, out_features, bias=False)

        nn.init.normal_(self.A.weight, std=0.02)
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        return self.B(self.A(x))


def apply_lora(model, rank=8):
    modules = list(model.named_modules())
    for name, module in modules:
        if isinstance(module, nn.Linear):

            lora = LoRA(
                module.in_features,
                module.out_features,
                rank=rank
            ).to(module.weight.device)

            module.lora = lora
            module._original_forward = module.forward

            def make_forward(module):
                def forward(x):
                    return module._original_forward(x) + module.lora(x)
                return forward

            module.forward = make_forward(module)

# DATA

In [169]:
from __future__ import annotations

from dataclasses import dataclass
from functools import partial
from typing import TypedDict

import torch
from datasets import Dataset as HFDataset
from datasets import DatasetDict, load_from_disk
from torch import Tensor
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import BatchSampler, DataLoader, Dataset, get_worker_info


# =========================
# Config
# =========================
@dataclass(slots=True)
class Seq2SeqDataConfig:
    dataset_path: str
    split: str = "training"

    max_source_length: int = 512
    max_target_length: int = 512

    random_crop: bool = True
    crop_seed: int = 0

    sliding_window_stride: int | None = None

    length_bucketing: bool = False
    bucket_size_multiplier: int = 50

    num_workers: int = 0

    pad_token_id: int = 0
    bos_token_id: int = 1
    eos_token_id: int = 2


# =========================
# Example / Batch
# =========================
class Seq2SeqExample(TypedDict):
    encoder_tokens: Tensor
    decoder_tokens: Tensor


@dataclass(slots=True)
class Seq2SeqBatch:
    encoder_input_tokens: Tensor
    encoder_padding_mask: Tensor
    decoder_input_tokens: Tensor
    labels: Tensor

    def to(self, device):
        return Seq2SeqBatch(
            encoder_input_tokens=self.encoder_input_tokens.to(device),
            encoder_padding_mask=self.encoder_padding_mask.to(device),
            decoder_input_tokens=self.decoder_input_tokens.to(device),
            labels=self.labels.to(device),
        )


# =========================
# Dataset
# =========================
class HuggingFaceSeq2SeqDataset(Dataset[Seq2SeqExample]):
    def __init__(self, config: Seq2SeqDataConfig):
        self.config = config

        dataset_dict = load_from_disk(config.dataset_path)
        if not isinstance(dataset_dict, DatasetDict):
            raise ValueError("dataset_path must point to DatasetDict")

        self.dataset: HFDataset = dataset_dict[config.split]

        self._crop_generators: dict[int, torch.Generator] = {}
        self._window_index: list[tuple[int, int]] | None = None

        if config.sliding_window_stride is not None:
            self._window_index = self._build_window_index()

    def __len__(self):
        return len(self._window_index) if self._window_index else len(self.dataset)

    def __getitem__(self, index: int) -> Seq2SeqExample:
        raw_index, window_start = self._resolve_index(index)

        item = self.dataset[raw_index]
        midi = item["midi_clean_ids"]
        lmx = item["lmx_ids"]

        # ✅ 对齐
        length = min(len(midi), len(lmx))
        midi = midi[:length]
        lmx = lmx[:length]

        # ✅ 先 trim（为 EOS 留空间）
        midi = self._trim(midi, window_start, self.config.max_source_length)
        lmx = self._trim(lmx, window_start, self.config.max_target_length - 1)

        # ✅ 再加 EOS（保证不被裁掉）
        lmx = lmx + [self.config.eos_token_id]

        return {
            "encoder_tokens": torch.tensor(midi, dtype=torch.long),
            "decoder_tokens": torch.tensor(lmx, dtype=torch.long),
        }

    def _resolve_index(self, index):
        if self._window_index is None:
            return index, None
        return self._window_index[index]

    def _trim(self, seq, start, max_len):
        if len(seq) <= max_len:
            return seq

        if start is not None:
            return seq[start:start + max_len]

        if self.config.random_crop and self.config.split == "training":
            max_start = len(seq) - max_len
            start = int(
                torch.randint(
                    0,
                    max_start + 1,
                    (1,),
                    generator=self._get_crop_generator()
                )
            )
            return seq[start:start + max_len]

        return seq[:max_len]

    def _build_window_index(self):
        stride = self.config.sliding_window_stride
        windows = []

        for i in range(len(self.dataset)):
            midi = self.dataset[i]["midi_clean_ids"]
            lmx = self.dataset[i]["lmx_ids"]

            length = min(len(midi), len(lmx))

            if length <= self.config.max_target_length:
                windows.append((i, 0))
                continue

            max_start = length - self.config.max_target_length
            starts = list(range(0, max_start + 1, stride))
            if starts[-1] != max_start:
                starts.append(max_start)

            windows.extend((i, s) for s in starts)

        return windows

    def _get_crop_generator(self):
        worker_info = get_worker_info()
        worker_id = worker_info.id if worker_info else 0

        generator = self._crop_generators.get(worker_id)
        if generator is None:
            generator = torch.Generator()
            generator.manual_seed(self.config.crop_seed + 1_000_003 * worker_id)
            self._crop_generators[worker_id] = generator

        return generator


# =========================
# Length Bucketing（完整补上）
# =========================
class LengthBucketBatchSampler(BatchSampler):
    def __init__(self, *, dataset, batch_size, drop_last, seed, bucket_size_multiplier):
        self.dataset = dataset
        self.batch_size = batch_size
        self.drop_last = drop_last
        self.seed = seed
        self.bucket_size_multiplier = bucket_size_multiplier
        self._epoch = 0

    def __iter__(self):
        g = torch.Generator().manual_seed(self.seed + self._epoch)
        self._epoch += 1

        indices = torch.randperm(len(self.dataset), generator=g).tolist()
        bucket_size = self.batch_size * self.bucket_size_multiplier

        batches = []
        for i in range(0, len(indices), bucket_size):
            pool = indices[i:i + bucket_size]
            for j in range(0, len(pool), self.batch_size):
                batch = pool[j:j + self.batch_size]
                if len(batch) == self.batch_size or not self.drop_last:
                    batches.append(batch)

        for idx in torch.randperm(len(batches), generator=g):
            yield batches[idx]

    def __len__(self):
        if self.drop_last:
            return len(self.dataset) // self.batch_size
        return (len(self.dataset) + self.batch_size - 1) // self.batch_size


# =========================
# Collate
# =========================
def collate_seq2seq_batch(
    examples: list[Seq2SeqExample],
    *,
    pad_token_id: int,
    bos_token_id: int,
) -> Seq2SeqBatch:

    encoder_tokens = pad_sequence(
        [e["encoder_tokens"] for e in examples],
        batch_first=True,
        padding_value=pad_token_id,
    )

    decoder_tokens = pad_sequence(
        [e["decoder_tokens"] for e in examples],
        batch_first=True,
        padding_value=pad_token_id,
    )

    # shift right
    decoder_input = decoder_tokens.clone()
    decoder_input[:, 1:] = decoder_tokens[:, :-1]
    decoder_input[:, 0] = bos_token_id

    labels = decoder_tokens.clone()
    labels[labels == pad_token_id] = -100

    # ✅ 更安全 mask（针对 [T,7]）
    encoder_mask = encoder_tokens[..., 0].eq(pad_token_id)

    return Seq2SeqBatch(
        encoder_input_tokens=encoder_tokens,
        encoder_padding_mask=encoder_mask,
        decoder_input_tokens=decoder_input,
        labels=labels,
    )


# =========================
# DataLoader
# =========================
def build_seq2seq_dataloader(
    config: Seq2SeqDataConfig,
    *,
    batch_size: int,
    shuffle: bool | None = None,
):
    dataset = HuggingFaceSeq2SeqDataset(config)

    if shuffle is None:
        shuffle = config.split == "training"

    dataloader_kwargs = {
        "dataset": dataset,
        "num_workers": config.num_workers,
        "collate_fn": partial(
            collate_seq2seq_batch,
            pad_token_id=config.pad_token_id,
            bos_token_id=config.bos_token_id,
        ),
    }

    if config.split == "training" and config.length_bucketing:
        dataloader_kwargs["batch_sampler"] = LengthBucketBatchSampler(
            dataset=dataset,
            batch_size=batch_size,
            drop_last=False,
            seed=config.crop_seed,
            bucket_size_multiplier=config.bucket_size_multiplier,
        )
    else:
        generator = None
        if config.split == "training":
            generator = torch.Generator().manual_seed(config.crop_seed)

        dataloader_kwargs["batch_size"] = batch_size
        dataloader_kwargs["shuffle"] = shuffle
        dataloader_kwargs["generator"] = generator

    return DataLoader(**dataloader_kwargs)

# Train

In [170]:
from __future__ import annotations

import csv
import time
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.nn.utils import clip_grad_norm_
from torch.optim import AdamW, Optimizer
from torch.optim.lr_scheduler import LambdaLR, LRScheduler
from torch.utils.tensorboard import SummaryWriter

# from midi2score.data_seq2seq import Seq2SeqDataConfig, build_seq2seq_dataloader
# from midi2score.model_seq2seq import Seq2SeqConfig, TransformerForConditionalGeneration


@dataclass(slots=True)
class Seq2SeqTrainingConfig:
    pretrained_decoder_path: str | None = None
    batch_size: int = 8
    learning_rate: float = 1e-4
    weight_decay: float = 0.01
    grad_clip_norm: float | None = 1.0
    label_smoothing: float = 0.0

    scheduler: str = "none"   # none / linear / cosine
    warmup_steps: int = 0
    min_lr_ratio: float = 0.0

    num_steps: int = 1000
    max_duration_seconds: float | None = None

    early_stopping_patience: int | None = None
    early_stopping_min_delta: float = 0.0

    log_every: int = 10
    eval_every: int = 0
    num_eval_batches: int | None = None

    device: str = "auto"

    save_checkpoint_path: str | None = None
    save_best_checkpoint_path: str | None = None
    resume_checkpoint_path: str | None = None

    csv_log_path: str | None = None
    tensorboard_log_dir: str | None = None

    freeze_encoder: bool = False
    freeze_decoder: bool = False

    def __post_init__(self) -> None:
        if self.batch_size <= 0:
            raise ValueError("batch_size must be positive.")
        if self.learning_rate <= 0:
            raise ValueError("learning_rate must be positive.")
        if self.weight_decay < 0:
            raise ValueError("weight_decay must be non-negative.")
        if self.grad_clip_norm is not None and self.grad_clip_norm <= 0:
            raise ValueError("grad_clip_norm must be positive.")
        if not 0.0 <= self.label_smoothing < 1.0:
            raise ValueError("label_smoothing must be in [0, 1).")
        if self.scheduler not in {"none", "linear", "cosine"}:
            raise ValueError("scheduler must be one of none/linear/cosine.")
        if self.warmup_steps < 0:
            raise ValueError("warmup_steps must be non-negative.")
        if not 0.0 <= self.min_lr_ratio <= 1.0:
            raise ValueError("min_lr_ratio must be between 0 and 1.")
        if self.num_steps <= 0:
            raise ValueError("num_steps must be positive.")
        if self.max_duration_seconds is not None and self.max_duration_seconds <= 0:
            raise ValueError("max_duration_seconds must be positive.")
        if self.early_stopping_patience is not None and self.early_stopping_patience <= 0:
            raise ValueError("early_stopping_patience must be positive.")
        if self.early_stopping_patience is not None and self.eval_every == 0:
            raise ValueError("early_stopping_patience requires eval_every > 0.")
        if self.num_eval_batches is not None and self.num_eval_batches <= 0:
            raise ValueError("num_eval_batches must be positive.")
        if self.resume_checkpoint_path is not None and not Path(self.resume_checkpoint_path).exists():
            raise ValueError(f"resume_checkpoint_path does not exist: {self.resume_checkpoint_path}")

    def to_dict(self) -> dict:
        return asdict(self)


@dataclass(slots=True)
class Seq2SeqEvaluationMetrics:
    loss: float
    token_accuracy: float
    top5_accuracy: float
    evaluated_tokens: int


@dataclass(slots=True)
class Seq2SeqTrainingResult:
    losses: list[float]
    validation_losses: list[tuple[int, float]]
    device: str
    checkpoint_path: str | None
    best_checkpoint_path: str | None
    best_validation_loss: float | None
    final_step: int
    resumed_from_checkpoint: str | None
    optimizer_state_loaded: bool
    elapsed_seconds: float
    stopped_due_to_time_budget: bool
    stopped_due_to_early_stopping: bool


@dataclass(slots=True)
class ResumeState:
    start_step: int
    best_validation_loss: float | None
    optimizer_loaded: bool
    scheduler_loaded: bool


@dataclass(slots=True)
class TrainingLogger:
    csv_path: str | None = None
    tensorboard_log_dir: str | None = None
    _writer: SummaryWriter | None = field(init=False, default=None, repr=False)

    def __post_init__(self) -> None:
        if self.csv_path is not None:
            csv_file = Path(self.csv_path)
            csv_file.parent.mkdir(parents=True, exist_ok=True)
            if not csv_file.exists():
                with csv_file.open("w", newline="", encoding="utf-8") as handle:
                    csv.writer(handle).writerow(["step", "split", "metric", "value"])
        if self.tensorboard_log_dir is not None:
            log_dir = Path(self.tensorboard_log_dir)
            log_dir.mkdir(parents=True, exist_ok=True)
            self._writer = SummaryWriter(log_dir=str(log_dir))

    def log_scalar(self, *, step: int, split: str, value: float, metric: str = "loss") -> None:
        if self.csv_path is not None:
            with Path(self.csv_path).open("a", newline="", encoding="utf-8") as handle:
                csv.writer(handle).writerow([step, split, metric, f"{value:.8f}"])
        if self._writer is not None:
            self._writer.add_scalar(f"{metric}/{split}", value, global_step=step)

    def close(self) -> None:
        if self._writer is not None:
            self._writer.close()


def resolve_device(requested_device: str) -> str:
    if requested_device != "auto":
        return requested_device
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def save_checkpoint(path: str, payload: dict) -> None:
    checkpoint_path = Path(path)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, checkpoint_path)


def load_checkpoint_for_resume(
    checkpoint_path: str,
    *,
    model,
    optimizer: Optimizer,
    scheduler: LRScheduler | None = None,
) -> ResumeState:
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(checkpoint["model_state"], strict=True)

    optimizer_loaded = False
    if "optimizer_state" in checkpoint and checkpoint["optimizer_state"] is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        optimizer_loaded = True

    scheduler_loaded = False
    if scheduler is not None and "scheduler_state" in checkpoint and checkpoint["scheduler_state"] is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state"])
        scheduler_loaded = True

    return ResumeState(
        start_step=int(checkpoint.get("step", 0)),
        best_validation_loss=checkpoint.get("best_validation_loss", checkpoint.get("validation_loss")),
        optimizer_loaded=optimizer_loaded,
        scheduler_loaded=scheduler_loaded,
    )


def build_lr_scheduler(optimizer: AdamW, training_config: Seq2SeqTrainingConfig) -> LambdaLR | None:
    if training_config.scheduler == "none" and training_config.warmup_steps == 0:
        return None

    total_steps = training_config.num_steps
    warmup_steps = training_config.warmup_steps
    min_lr_ratio = training_config.min_lr_ratio
    scheduler_name = training_config.scheduler

    def lr_lambda(step: int) -> float:
        current_step = max(step, 1)

        if warmup_steps > 0 and current_step <= warmup_steps:
            return current_step / warmup_steps

        if scheduler_name == "none":
            return 1.0

        decay_start = max(warmup_steps, 1)
        decay_steps = max(total_steps - decay_start, 1)
        progress = min(max((current_step - decay_start) / decay_steps, 0.0), 1.0)

        if scheduler_name == "linear":
            return 1.0 - progress * (1.0 - min_lr_ratio)

        cosine = 0.5 * (1.0 + torch.cos(torch.tensor(progress * torch.pi))).item()
        return min_lr_ratio + (1.0 - min_lr_ratio) * cosine

    return LambdaLR(optimizer, lr_lambda=lr_lambda)


def freeze_modules(model: TransformerForConditionalGeneration, config: Seq2SeqTrainingConfig) -> None:
    if config.freeze_encoder:
        for p in model.model.encoder.parameters():
            p.requires_grad = False

    if config.freeze_decoder:
        for p in model.model.decoder.parameters():
            p.requires_grad = False


def evaluate_seq2seq_metrics(
    model: TransformerForConditionalGeneration,
    loader,
    *,
    device: torch.device | str,
    num_batches: int | None = None,
) -> Seq2SeqEvaluationMetrics:
    model.eval()

    total_loss = 0.0
    total_batches = 0
    total_valid_tokens = 0
    total_correct_tokens = 0
    total_top5_correct_tokens = 0

    with torch.no_grad():
        for batch_index, batch in enumerate(loader, start=1):
            batch = batch.to(device)

            loss, logits = model(
                encoder_input_tokens=batch.encoder_input_tokens,
                decoder_input_tokens=batch.decoder_input_tokens,
                labels=batch.labels,
                encoder_padding_mask=batch.encoder_padding_mask,
            )

            total_loss += float(loss.item())
            total_batches += 1

            flat_labels = batch.labels.reshape(-1)
            valid_mask = flat_labels.ne(-100)
            valid_labels = flat_labels[valid_mask]

            if valid_labels.numel() > 0:
                flat_logits = logits.reshape(-1, logits.size(-1))[valid_mask]
                predictions = flat_logits.argmax(dim=-1)
                topk = flat_logits.topk(k=min(5, flat_logits.size(-1)), dim=-1).indices

                total_valid_tokens += int(valid_labels.numel())
                total_correct_tokens += int(predictions.eq(valid_labels).sum().item())
                total_top5_correct_tokens += int(
                    topk.eq(valid_labels.unsqueeze(-1)).any(dim=-1).sum().item()
                )

            if num_batches is not None and batch_index >= num_batches:
                break

    avg_loss = total_loss / max(total_batches, 1)
    token_accuracy = total_correct_tokens / total_valid_tokens if total_valid_tokens > 0 else 0.0
    top5_accuracy = total_top5_correct_tokens / total_valid_tokens if total_valid_tokens > 0 else 0.0

    return Seq2SeqEvaluationMetrics(
        loss=avg_loss,
        token_accuracy=token_accuracy,
        top5_accuracy=top5_accuracy,
        evaluated_tokens=total_valid_tokens,
    )

def load_pretrained_decoder(seq2seq_model, checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location="cpu")
    pretrained_dict = ckpt["model_state"]

    model_dict = seq2seq_model.state_dict()

    loaded = []
    skipped = []

    for k, v in pretrained_dict.items():
        new_k =  "model.decoder." + k   # 🔥 核心

        if new_k in model_dict:
            if model_dict[new_k].shape == v.shape:
                model_dict[new_k] = v
                loaded.append(new_k)
            else:
                skipped.append((new_k, "shape mismatch"))
        else:
            skipped.append((new_k, "not found"))

    seq2seq_model.load_state_dict(model_dict)

    print(f"✅ Loaded: {len(loaded)}")
    print(f"⚠️ Skipped: {len(skipped)}")

    return loaded, skipped

def run_seq2seq_training_loop(
    model_config: Seq2SeqConfig,
    data_config: Seq2SeqDataConfig,
    training_config: Seq2SeqTrainingConfig,
) -> Seq2SeqTrainingResult:
    _validate_setup(model_config, data_config)

    device = resolve_device(training_config.device)

    train_loader = build_seq2seq_dataloader(
        data_config,
        batch_size=training_config.batch_size,
    )

    validation_loader = None
    if training_config.eval_every > 0:
        validation_loader = build_seq2seq_dataloader(
            replace(data_config, split="validation", random_crop=False),
            batch_size=training_config.batch_size,
            shuffle=False,
        )

    model = TransformerForConditionalGeneration(model_config).to(device)

    #加载pretrain参数
    if training_config.pretrained_decoder_path is not None:
        load_pretrained_decoder(model, training_config.pretrained_decoder_path)

    apply_lora(model.model.decoder)

    # ✅ Step 1: 先全部 freeze
    for p in model.parameters():
        p.requires_grad = False

    # ✅ Step 2: 打开 encoder
    for p in model.model.encoder.parameters():
        p.requires_grad = True

    # ✅ Step 3: 只打开 LoRA
    for name, p in model.named_parameters():
        if "lora" in name:
            p.requires_grad = True

    print("\n==== TRAINABLE PARAMS ====")
    for name, p in model.named_parameters():
        if p.requires_grad:
            print(name)
            
    trainable_parameters = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(
        trainable_parameters,
        lr=training_config.learning_rate,
        weight_decay=training_config.weight_decay,
    )
    scheduler = build_lr_scheduler(optimizer, training_config)

    logger = TrainingLogger(
        csv_path=training_config.csv_log_path,
        tensorboard_log_dir=training_config.tensorboard_log_dir,
    )

    losses: list[float] = []
    validation_losses: list[tuple[int, float]] = []
    start_step = 0
    best_validation_loss: float | None = None
    best_step = 0
    consecutive_non_improving_evals = 0
    optimizer_state_loaded = False

    if training_config.resume_checkpoint_path is not None:
        resume_state = load_checkpoint_for_resume(
            training_config.resume_checkpoint_path,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
        )
        start_step = resume_state.start_step
        best_validation_loss = resume_state.best_validation_loss
        best_step = start_step
        optimizer_state_loaded = resume_state.optimizer_loaded

    iterator = iter(train_loader)
    started_at = time.monotonic()
    final_step = start_step
    stopped_due_to_time_budget = False
    stopped_due_to_early_stopping = False

    try:
        for step in range(start_step + 1, training_config.num_steps + 1):
            if (
                training_config.max_duration_seconds is not None
                and step > start_step + 1
                and time.monotonic() - started_at >= training_config.max_duration_seconds
            ):
                stopped_due_to_time_budget = True
                break

            try:
                batch = next(iterator)
            except StopIteration:
                iterator = iter(train_loader)
                batch = next(iterator)

            batch = batch.to(device)

            model.train()
            optimizer.zero_grad(set_to_none=True)

            loss, logits = model(
                encoder_input_tokens=batch.encoder_input_tokens,
                decoder_input_tokens=batch.decoder_input_tokens,
                labels=batch.labels,
                encoder_padding_mask=batch.encoder_padding_mask,
            )

            if training_config.label_smoothing > 0.0:
                flat_logits = logits.reshape(-1, logits.size(-1))
                flat_labels = batch.labels.reshape(-1)
                loss = F.cross_entropy(
                    flat_logits,
                    flat_labels,
                    ignore_index=-100,
                    label_smoothing=training_config.label_smoothing,
                )

            loss.backward()

            if training_config.grad_clip_norm is not None:
                clip_grad_norm_(trainable_parameters, max_norm=training_config.grad_clip_norm)

            optimizer.step()
            if scheduler is not None:
                scheduler.step()

            loss_value = float(loss.item())
            losses.append(loss_value)
            final_step = step

            logger.log_scalar(step=step, split="train", value=loss_value)

            if step % training_config.log_every == 0:
                print(f"step={step} seq2seq_train_loss={loss_value:.4f} device={device}")

            if validation_loader is not None and step % training_config.eval_every == 0:
                metrics = evaluate_seq2seq_metrics(
                    model,
                    validation_loader,
                    device=device,
                    num_batches=training_config.num_eval_batches,
                )

                validation_losses.append((step, metrics.loss))
                logger.log_scalar(step=step, split="validation", value=metrics.loss)
                logger.log_scalar(step=step, split="validation", metric="token_accuracy", value=metrics.token_accuracy)
                logger.log_scalar(step=step, split="validation", metric="top5_accuracy", value=metrics.top5_accuracy)

                print(
                    f"step={step} validation_loss={metrics.loss:.4f} "
                    f"token_acc={metrics.token_accuracy:.4f} "
                    f"top5_acc={metrics.top5_accuracy:.4f} "
                    f"device={device}"
                )

                improved = best_validation_loss is None or (
                    metrics.loss < best_validation_loss - training_config.early_stopping_min_delta
                )

                if improved:
                    best_validation_loss = metrics.loss
                    best_step = step
                    consecutive_non_improving_evals = 0

                    if training_config.save_best_checkpoint_path is not None:
                        save_checkpoint(
                            training_config.save_best_checkpoint_path,
                            _checkpoint_payload(
                                model=model,
                                optimizer=optimizer,
                                scheduler=scheduler,
                                model_config=model_config,
                                data_config=data_config,
                                training_config=training_config,
                                step=step,
                                best_validation_loss=best_validation_loss,
                            ),
                        )
                else:
                    consecutive_non_improving_evals += 1

                if (
                    training_config.early_stopping_patience is not None
                    and consecutive_non_improving_evals >= training_config.early_stopping_patience
                ):
                    stopped_due_to_early_stopping = True
                    print(
                        "early_stopping_triggered "
                        f"step={step} best_step={best_step} "
                        f"best_validation_loss={best_validation_loss:.4f}"
                    )
                    break
    finally:
        logger.close()

    elapsed_seconds = time.monotonic() - started_at

    if training_config.save_checkpoint_path is not None:
        payload = _checkpoint_payload(
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            model_config=model_config,
            data_config=data_config,
            training_config=training_config,
            step=final_step,
            best_validation_loss=best_validation_loss,
        )
        payload["elapsed_seconds"] = elapsed_seconds
        payload["stopped_due_to_time_budget"] = stopped_due_to_time_budget
        payload["stopped_due_to_early_stopping"] = stopped_due_to_early_stopping
        save_checkpoint(training_config.save_checkpoint_path, payload)

    return Seq2SeqTrainingResult(
        losses=losses,
        validation_losses=validation_losses,
        device=device,
        checkpoint_path=training_config.save_checkpoint_path,
        best_checkpoint_path=training_config.save_best_checkpoint_path,
        best_validation_loss=best_validation_loss,
        final_step=final_step,
        resumed_from_checkpoint=training_config.resume_checkpoint_path,
        optimizer_state_loaded=optimizer_state_loaded,
        elapsed_seconds=elapsed_seconds,
        stopped_due_to_time_budget=stopped_due_to_time_budget,
        stopped_due_to_early_stopping=stopped_due_to_early_stopping,
    )


def _checkpoint_payload(
    *,
    model: TransformerForConditionalGeneration,
    optimizer: AdamW,
    scheduler: LambdaLR | None,
    model_config: Seq2SeqConfig,
    data_config: Seq2SeqDataConfig,
    training_config: Seq2SeqTrainingConfig,
    step: int,
    best_validation_loss: float | None,
) -> dict:
    return {
        "model_type": "seq2seq",
        "model_config": model_config.to_dict(),
        "data_config": data_config.__dict__,
        "training_config": training_config.to_dict(),
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": None if scheduler is None else scheduler.state_dict(),
        "step": step,
        "validation_loss": best_validation_loss,
        "best_validation_loss": best_validation_loss,
    }


def _validate_setup(
    model_config: Seq2SeqConfig,
    data_config: Seq2SeqDataConfig,
) -> None:
    if model_config.max_source_length < data_config.max_source_length:
        raise ValueError("model.max_source_length must be >= data.max_source_length.")
    if model_config.max_target_length < data_config.max_target_length:
        raise ValueError("model.max_target_length must be >= data.max_target_length.")

In [1]:
from datasets import load_from_disk

dataset = load_from_disk("DATA/huggingface_seq2seq_rd")
dataset

/Users/hao/miniconda3/envs/csci544_hw1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    training: Dataset({
        features: ['lmx_ids', 'midi_clean_ids', 'midi_light_ids', 'midi_heavy_ids'],
        num_rows: 29514
    })
    validation: Dataset({
        features: ['lmx_ids', 'midi_clean_ids', 'midi_light_ids', 'midi_heavy_ids'],
        num_rows: 7308
    })
    test: Dataset({
        features: ['lmx_ids', 'midi_clean_ids', 'midi_light_ids', 'midi_heavy_ids'],
        num_rows: 9086
    })
})

In [19]:
def check_lengths(dataset, split="training", num_samples=100):
    for key in dataset[split].column_names:
        lengths = [len(dataset[split][i][key]) for i in range(num_samples)]
        
        print(f"\n=== {key} ===")
        print("min:", min(lengths))
        print("max:", max(lengths))
        print("avg:", sum(lengths) / len(lengths))

check_lengths(dataset)


=== lmx_ids ===
min: 44
max: 1746
avg: 309.81

=== midi_clean_ids ===
min: 2
max: 2067
avg: 419.83

=== midi_light_ids ===
min: 2
max: 2175
avg: 430.58

=== midi_heavy_ids ===
min: 2
max: 2285
avg: 442.96


In [23]:
sample = dataset["training"][1]
sample

{'lmx_ids': [1,
  647,
  591,
  911,
  2468,
  561,
  353,
  489,
  1458,
  287,
  4500,
  180,
  726,
  466,
  261,
  4500,
  726,
  466,
  261,
  227,
  263,
  3340,
  561,
  4656,
  2681,
  779,
  482,
  315,
  489,
  482,
  315,
  263,
  2468,
  561,
  717,
  300,
  2681,
  920,
  266,
  726,
  1981,
  412,
  726,
  1575,
  466,
  455,
  625,
  1703,
  320,
  4568,
  513,
  4328,
  455,
  625,
  1109,
  412,
  726,
  4983,
  3052,
  1184,
  1899,
  1039,
  1899,
  1184,
  1946,
  676,
  1460,
  412,
  726,
  1575,
  466,
  455,
  625,
  1109,
  412,
  726,
  1575,
  466,
  455,
  625,
  1109,
  412,
  726,
  4983,
  3052,
  1184,
  1899,
  1039,
  489,
  4716,
  1946,
  676,
  1654,
  1981,
  387,
  701,
  1164,
  453,
  2932,
  1074,
  1866,
  263,
  2345,
  175,
  561,
  353,
  2533,
  287,
  4500,
  726,
  466,
  261,
  4500,
  726,
  466,
  1174,
  1384,
  507,
  360,
  3532,
  353,
  1517,
  642,
  204,
  1962,
  455,
  235,
  169,
  279,
  719,
  880,
  387,
  293,
  845,
  2

In [24]:
for k, v in sample.items():
    print(k, type(v), len(v))

lmx_ids <class 'list'> 222
midi_clean_ids <class 'list'> 506
midi_light_ids <class 'list'> 506
midi_heavy_ids <class 'list'> 506


In [16]:
midi = sample["midi_clean_ids"]

print(len(midi))        # T
print(len(midi[0]))     # 7

506
7


In [171]:
data_config = Seq2SeqDataConfig(
    dataset_path="./DATA/huggingface_seq2seq_rd",  # ⚠️路径改这里
    split="training",

    max_source_length=128,   # ⚠️先缩短！！
    max_target_length=128,

    num_workers=0,
)

In [172]:
training_config = Seq2SeqTrainingConfig(
    batch_size=2,        # ⚠️小
    num_steps=20,        # ⚠️小测试
    log_every=1,
    eval_every=0,
    pretrained_decoder_path="MIDI2Score/best_models/rd/best.pt",
    device="cuda" if torch.cuda.is_available() else "cpu",
)

In [173]:
model_config = Seq2SeqConfig(
    src_vocab_size=1000,   # 👈 随便填（不会用到）

    tgt_vocab_size=5000,
    d_model=256,
    nhead=4,
    num_decoder_layers=2,
    dim_feedforward=1024,

    src_vocab_size_list=[128]*7,
    src_embedding_size_list=[16]*7,

    num_encoder_layers=2,
)

In [174]:
result = run_seq2seq_training_loop(
    model_config=model_config,
    data_config=data_config,
    training_config=training_config,
)

✅ Loaded: 41
⚠️ Skipped: 0

==== TRAINABLE PARAMS ====
model.encoder.embedding.family_embedding.weight
model.encoder.embedding.position_embedding.weight
model.encoder.embedding.pitch_embedding.weight
model.encoder.embedding.duration_embedding.weight
model.encoder.embedding.program_embedding.weight
model.encoder.embedding.tempo_embedding.weight
model.encoder.embedding.time_signature_embedding.weight
model.encoder.embedding.in_linear.weight
model.encoder.embedding.in_linear.bias
model.encoder.encoder.layers.0.self_attn.in_proj_weight
model.encoder.encoder.layers.0.self_attn.in_proj_bias
model.encoder.encoder.layers.0.self_attn.out_proj.weight
model.encoder.encoder.layers.0.self_attn.out_proj.bias
model.encoder.encoder.layers.0.linear1.weight
model.encoder.encoder.layers.0.linear1.bias
model.encoder.encoder.layers.0.linear2.weight
model.encoder.encoder.layers.0.linear2.bias
model.encoder.encoder.layers.0.norm1.weight
model.encoder.encoder.layers.0.norm1.bias
model.encoder.encoder.layers.0

In [165]:
model = TransformerForConditionalGeneration(model_config)

loaded, skipped = load_pretrained_decoder(
    model,
    "MIDI2Score/best_models/rd/best.pt"
)

✅ Loaded: 41
⚠️ Skipped: 0
